# Explorer une édition TEI (Cligés de Chrétien de Troyes)

Notebook pour exploiter une édition TEI :
- inventaire témoins/types
- cartes couleur HTML (densité globale et par type)
- zoom sur un vers (lecture par témoin + apparats groupés)
- profils de témoins (matrice témoin × type)
- matrices proximité/divergence
- exports TSV/HTML

## 0) Réglages & chemins
Notebook à placer dans `collazione/`.
Fichier TEI attendu : `../data/tei/cliges_unifier/cliges_unified_fixed_unique.xml`.

In [5]:
from pathlib import Path
import os, re
from lxml import etree
from collections import Counter, defaultdict
from IPython.display import HTML, display

# --- Fixe la racine du projet (dossier qui contient data/) ---
here = Path().resolve()
root = here
if not (root / "data").exists() and (root.parent / "data").exists():
    root = root.parent
else:
    while root != root.parent and not (root / "data").exists():
        root = root.parent
if not (root / "data").exists():
    raise RuntimeError(f"Impossible de trouver la racine (dossier data/) depuis {here}")

os.chdir(root)
print("✅ CWD fixé à la racine :", Path().resolve())

TEI_PATH = Path("data/tei/cliges_unified/cliges_unified_fixed_unique.xml")
print("Fichier TEI :", TEI_PATH)
print("Existe ?", TEI_PATH.exists())

TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"

EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)


✅ CWD fixé à la racine : /Users/benedettasalvati/Desktop/Workshop2026_07-main
Fichier TEI : data/tei/cliges_unified/cliges_unified_fixed_unique.xml
Existe ? True


## 1) Charger le TEI + inventaire (témoins, vers, apparats, types)

In [6]:
#pour retrouver l'emplacement du fichier
from pathlib import Path
print("CWD =", Path().resolve())

for p in Path("data").rglob("*"):
    if "cliges" in p.name.lower():
        print(p)

CWD = /Users/benedettasalvati/Desktop/Workshop2026_07-main
data/tei/cliges_lemma
data/tei/cliges
data/tei/cliges_unified
data/tei/cliges_lemma/Cliges_A.xml
data/tei/cliges_lemma/Cliges_P.xml
data/tei/cliges_unified/cliges_unified_fixed_unique.xml


In [7]:
parser = etree.XMLParser(recover=True, huge_tree=True)
tree = etree.parse(str(TEI_PATH), parser)
root = tree.getroot()
from pathlib import Path
print("CWD =", Path().resolve())

for p in Path("data").rglob("*"):
    if "cliges" in p.name.lower():
        print(p)
witness_nodes = root.xpath(".//tei:listWit//tei:witness", namespaces=TEI_NS)
witness_ids = []
for w in witness_nodes:
    wid = w.get(XML_ID) or w.get("n") or (w.text or "").strip()
    if wid:
        witness_ids.append(wid.strip())

line_nodes = root.xpath(".//tei:l", namespaces=TEI_NS)
app_nodes  = root.xpath(".//tei:app", namespaces=TEI_NS)
rdg_nodes  = root.xpath(".//tei:rdg", namespaces=TEI_NS)

print("Témoins (listWit) :", witness_ids)
print("Nombre de vers (<l>) :", len(line_nodes))
print("Nombre d'apparats (<app>) :", len(app_nodes))
print("Nombre de lectures (<rdg>) :", len(rdg_nodes))

type_counts_raw = Counter(a.get("type","(none)") for a in app_nodes)
print("\nTypes d'apparat (top 25) :")
for t, n in type_counts_raw.most_common(25):
    print(f"{t:>20}  {n}")


CWD = /Users/benedettasalvati/Desktop/Workshop2026_07-main
data/tei/cliges_lemma
data/tei/cliges
data/tei/cliges_unified
data/tei/cliges_lemma/Cliges_A.xml
data/tei/cliges_lemma/Cliges_P.xml
data/tei/cliges_unified/cliges_unified_fixed_unique.xml
Témoins (listWit) : ['A', 'B', 'P', 'C', 'R', 'S', 'T', 'M']
Nombre de vers (<l>) : 1289
Nombre d'apparats (<app>) : 6617
Nombre de lectures (<rdg>) : 12774

Types d'apparat (top 25) :
           graphemic  2630
     morphosyntactic  811
       lexsem:strong  747
             absence  655
         lexsem:weak  649
              (none)  467
         morphologic  284
               order  229
     lexsem:nonsense  145


### 1b) Normaliser les types (`@type`)

In [8]:
def norm_type(t: str) -> str:
    if not t:
        return "(none)"
    s = t.strip().lower()
    s = s.lstrip(",; ")
    s = re.sub(r"\s+", " ", s)
    s = s.replace("morphological", "morphologic")
    s = s.replace("morphosyntactical", "morphosyntactic")
    return s

TYPE_MAP = {
    "lexsem": "lexsem",
    "graphemic": "graphemic",
    "morphologic": "morphologic",
    "morphosyntactic": "morphosyntactic",
    "": "(none)",
}

def canon_type(t: str) -> str:
    s = norm_type(t)
    return TYPE_MAP.get(s, s)

type_counts = Counter(canon_type(a.get("type","(none)")) for a in app_nodes)

html = "<h3>Types normalisés</h3><table style='border-collapse:collapse'>"
html += "<tr><th style='border:1px solid #ddd;padding:6px'>type</th><th style='border:1px solid #ddd;padding:6px'>count</th></tr>"
for t, n in type_counts.most_common():
    html += f"<tr><td style='border:1px solid #ddd;padding:6px'><b>{t}</b></td><td style='border:1px solid #ddd;padding:6px;text-align:right'>{n}</td></tr>"
html += "</table>"
display(HTML(html))

tsv = EXPORTS / "cliges_types_apparat.tsv"
with tsv.open("w", encoding="utf-8") as f:
    f.write("type\tcount\n")
    for t, n in type_counts.most_common():
        f.write(f"{t}\t{n}\n")
print("✅ Export :", tsv)


type,count
graphemic,2630
morphosyntactic,811
lexsem:strong,747
absence,655
lexsem:weak,649
(none),467
morphologic,284
order,229
lexsem:nonsense,145


✅ Export : exports/cliges_types_apparat.tsv


In [9]:
# --- Filtrage des types rares (coquilles probables) ---

MIN_OCC = 10  # seuil : à ajuster si besoin

# types conservés
VALID_TYPES = {t for t, n in type_counts.items() if n >= MIN_OCC}

# types exclus
REMOVED_TYPES = {t: n for t, n in type_counts.items() if n < MIN_OCC}

print(f"Types conservés (≥ {MIN_OCC} occurrences) :", sorted(VALID_TYPES))
print(f"\nTypes exclus (< {MIN_OCC} occurrences) :")
for t, n in sorted(REMOVED_TYPES.items(), key=lambda x: x[1]):
    print(f"  - {t} : {n}")

# fonction utilitaire : dire si un <app> est valide
def is_valid_app(app):
    return canon_type(app.get("type", "(none)")) in VALID_TYPES

Types conservés (≥ 10 occurrences) : ['(none)', 'absence', 'graphemic', 'lexsem:nonsense', 'lexsem:strong', 'lexsem:weak', 'morphologic', 'morphosyntactic', 'order']

Types exclus (< 10 occurrences) :


## 2) Carte du texte : densité des variantes par vers (HTML)

In [10]:
apps_by_line = defaultdict(list)
for i, l in enumerate(line_nodes, start=1):
    apps_in_line = l.xpath(".//tei:app", namespaces=TEI_NS)
    if apps_in_line:
        apps_by_line[i].extend(apps_in_line)

density_all = Counter({i: len(apps_by_line.get(i, [])) for i in range(1, len(line_nodes)+1)})

print("Top 15 vers (densité d'apparat, tous types) :")
for ln, c in density_all.most_common(15):
    print(f"Vers {ln:>5} : {c}")

max_ln = len(line_nodes)
vec = [density_all.get(i, 0) for i in range(1, max_ln+1)]
mx = max(vec) if vec else 1
norm = [(v/mx if mx else 0) for v in vec]

def color_red(v):
    r = int(255*v)
    return f"rgb(255,{255-r},{255-r})"

cells = []
for i, v in enumerate(norm, start=1):
    cells.append(
        f"<div title='Vers {i}: {vec[i-1]} apparats' style='width:10px;height:18px;background:{color_red(v)};border:1px solid #eee;'></div>"
    )

card = "<h3>Carte couleur — densité d'apparat par vers (tous types)</h3>"
card += "<div style='display:flex;flex-wrap:wrap;max-width:980px'>" + "".join(cells) + "</div>"
card += "<p><small>Plus foncé = plus d'apparats. Survoler un bloc pour voir le vers et le nombre.</small></p>"
display(HTML(card))

html_doc = "<!doctype html><html><meta charset='utf-8'>"
html_doc += "<body style='font-family:system-ui;margin:24px'>"
html_doc += "<h2>Carte couleur — densité d'apparat par vers (tous types)</h2>"
html_doc += "<div style='display:flex;flex-wrap:wrap;max-width:980px'>" + "".join(cells) + "</div>"
html_doc += "<p><small>Imprimer : Ctrl/Cmd+P → Enregistrer en PDF.</small></p>"
html_doc += "</body></html>"
out = EXPORTS / "carte_densite_apparat_all.html"
out.write_text(html_doc, encoding="utf-8")
print("✅ Carte sauvegardée :", out)


Top 15 vers (densité d'apparat, tous types) :
Vers   203 : 28
Vers    77 : 16
Vers   226 : 15
Vers   858 : 15
Vers   703 : 14
Vers   886 : 14
Vers  1223 : 14
Vers    99 : 12
Vers   375 : 12
Vers   713 : 12
Vers   730 : 12
Vers   805 : 12
Vers   909 : 12
Vers  1093 : 12
Vers    72 : 11


✅ Carte sauvegardée : exports/carte_densite_apparat_all.html


### 2b) Carte par type (filtre)
Choisis un type et génère une carte dédiée.

In [11]:
import ipywidgets as widgets
from IPython.display import HTML, display
from collections import Counter

types_sorted = [t for t,_ in type_counts.most_common()]

def show_type_map(t):
    dens = Counter()
    for i in range(1, len(line_nodes)+1):
        dens[i] = sum(1 for a in apps_by_line.get(i, []) if canon_type(a.get("type","(none)")) == t)

    vec_t = [dens.get(i, 0) for i in range(1, len(line_nodes)+1)]
    mx_t = max(vec_t) if vec_t else 1
    norm_t = [(v/mx_t if mx_t else 0) for v in vec_t]

    cells_t = []
    for i, v in enumerate(norm_t, start=1):
        cells_t.append(
            f"<div title='Vers {i}: {vec_t[i-1]} apparats ({t})' "
            f"style='width:10px;height:18px;background:{color_red(v)};border:1px solid #eee;'></div>"
        )

    display(HTML(
        "<h3>Carte — type: <code>"+t+"</code></h3>"
        "<div style='display:flex;flex-wrap:wrap;max-width:980px'>"
        + "".join(cells_t) +
        "</div>"
    ))

widgets.interact(show_type_map, t=widgets.Dropdown(options=types_sorted, description="Type:"));

interactive(children=(Dropdown(description='Type:', options=('graphemic', 'morphosyntactic', 'lexsem:strong', …

## 3) Zoom sur un vers : lecture par témoin + apparats groupés
Reconstitue une lecture pour un témoin en sélectionnant la bonne `<rdg>` dans chaque `<app>`.

In [27]:
def parse_wit_list(attr: str):
    if not attr:
        return []
    return [w.strip().lstrip("#") for w in attr.split() if w.strip()]

def pick_reading(app, wit: str):
    for rdg in app.xpath("./tei:rdg", namespaces=TEI_NS):
        if wit in parse_wit_list(rdg.get("wit","")):
            return rdg
    for lem in app.xpath("./tei:lem", namespaces=TEI_NS):
        wits = parse_wit_list(lem.get("wit",""))
        if (not wits) or (wit in wits):
            return lem
    lem_nodes = app.xpath("./tei:lem", namespaces=TEI_NS)
    if lem_nodes:
        return lem_nodes[0]
    rdg_nodes = app.xpath("./tei:rdg", namespaces=TEI_NS)
    return rdg_nodes[0] if rdg_nodes else None

def text_of_node(node):
    if node is None:
        return ""
    return " ".join(" ".join(node.itertext()).split())

def render_line_for_wit(line_node, wit: str):
    parts = []
    for child in line_node.iterchildren():
        tag = etree.QName(child).localname
        if tag == "app":
            sel = pick_reading(child, wit)
            parts.append(text_of_node(sel))
        else:
            parts.append(text_of_node(child))
    return " ".join(" ".join(parts).split())

def apparatus_summary_for_line(line_node):
    by_type = defaultdict(list)
    for app in line_node.xpath(".//tei:app", namespaces=TEI_NS):
        t = canon_type(app.get("type","(none)"))
        lem = app.xpath("./tei:lem", namespaces=TEI_NS)
        lem_txt = text_of_node(lem[0]) if lem else ""
        rdgs = []
        for rdg in app.xpath("./tei:rdg", namespaces=TEI_NS):
            rdgs.append({"wit": " ".join(parse_wit_list(rdg.get("wit",""))), "txt": text_of_node(rdg)})
        by_type[t].append({"lem": lem_txt, "rdgs": rdgs})
    return by_type


In [28]:
import ipywidgets as widgets
from IPython.display import HTML, display

wit_opts = witness_ids if witness_ids else ["(unknown)"]

def show_line(line_no: int, wit: str):
    lnode = line_nodes[line_no-1]
    txt = render_line_for_wit(lnode, wit)
    by_type = apparatus_summary_for_line(lnode)

    html = f"<h3>Vers {line_no} — lecture <code>{wit}</code></h3>"
    html += f"<p style='font-size:1.1em'><b>{txt}</b></p>"

    if not by_type:
        html += "<p><i>Aucun apparat dans ce vers.</i></p>"
        display(HTML(html))
        return

    html += "<h4>Apparats (groupés par type)</h4>"
    for t in sorted(by_type.keys()):
        html += f"<h5><code>{t}</code> ({len(by_type[t])})</h5><ul>"
        for item in by_type[t]:
            html += f"<li><b>lem</b> : {item['lem']}"
            if item["rdgs"]:
                html += "<ul>"
                for r in item["rdgs"]:
                    html += f"<li><b>{r['wit']}</b> : {r['txt']}</li>"
                html += "</ul>"
            html += "</li>"
        html += "</ul>"
    display(HTML(html))

widgets.interact(
    show_line,
    line_no=widgets.IntSlider(value=1, min=1, max=len(line_nodes), step=1, description="Vers:"),
    wit=widgets.Dropdown(options=wit_opts, description="Témoin:")
);


interactive(children=(IntSlider(value=1, description='Vers:', max=1289, min=1), Dropdown(description='Témoin:'…

## 4) Profils de témoins : matrice témoin × type + export TSV

In [29]:
from collections import defaultdict, Counter
from IPython.display import HTML, display

matrix = defaultdict(Counter)
for app in app_nodes:
    t = canon_type(app.get("type", "(none)"))
    for el in app.xpath("./tei:lem | ./tei:rdg", namespaces=TEI_NS):
        for w in parse_wit_list(el.get("wit", "")):
            matrix[w][t] += 1

all_types = sorted({t for w in matrix for t in matrix[w]})
all_wits = sorted(matrix.keys()) if matrix else witness_ids

# --- affichage HTML ---
html = "<h3>Matrice témoin × type</h3><table style='border-collapse:collapse'>"
html += "<tr><th style='border:1px solid #ddd;padding:6px'>wit</th>"
for t in all_types:
    html += f"<th style='border:1px solid #ddd;padding:6px'>{t}</th>"
html += "<th style='border:1px solid #ddd;padding:6px'>total</th></tr>"

for w in all_wits:
    total = sum(matrix[w].values())
    html += f"<tr><td style='border:1px solid #ddd;padding:6px'><b>{w}</b></td>"
    for t in all_types:
        html += f"<td style='border:1px solid #ddd;padding:6px;text-align:right'>{matrix[w].get(t,0)}</td>"
    html += f"<td style='border:1px solid #ddd;padding:6px;text-align:right'><b>{total}</b></td></tr>"

html += "</table>"
display(HTML(html))

# --- export TSV ---
out = EXPORTS / "cliges_matrix_wit_type.tsv"
with out.open("w", encoding="utf-8") as f:
    f.write("wit\t" + "\t".join(all_types) + "\ttotal\n")
    for w in all_wits:
        total = sum(matrix[w].values())
        row = [str(matrix[w].get(t, 0)) for t in all_types]
        f.write(w + "\t" + "\t".join(row) + "\t" + str(total) + "\n")

print("✅ Export :", out)

wit,(none),absence,graphemic,lexsem:nonsense,lexsem:strong,lexsem:weak,morphologic,morphosyntactic,order,total
A,431,638,2346,134,727,618,261,769,215,6139
B,423,644,2307,139,732,619,266,754,219,6103
C,433,643,2413,144,738,634,265,779,214,6263
M,0,29,169,48,226,175,7,41,17,712
P,429,638,2284,134,720,630,250,746,214,6045
R,431,642,2339,143,726,625,262,763,211,6142
S,416,628,2017,145,710,602,243,722,216,5699
T,0,1,0,0,0,0,0,0,0,1


✅ Export : exports/cliges_matrix_wit_type.tsv


## 5) Matrices proximité/divergence entre témoins + export TSV

In [30]:
from itertools import combinations
from collections import defaultdict, Counter
from IPython.display import HTML, display

coocc = defaultdict(Counter)
div = defaultdict(Counter)

for app in app_nodes:
    groups = []
    for rdg in app.xpath("./tei:rdg | ./tei:lem", namespaces=TEI_NS):
        wits = sorted(set(parse_wit_list(rdg.get("wit", ""))))
        if wits:
            groups.append(wits)

    # co-occurrence : même lecture
    for g in groups:
        for a, b in combinations(g, 2):
            coocc[a][b] += 1
            coocc[b][a] += 1

    # divergence : lectures différentes dans le même app
    assign = {}
    for idx, g in enumerate(groups):
        for w in g:
            assign[w] = idx

    wits_present = sorted(assign.keys())
    for a, b in combinations(wits_present, 2):
        if assign[a] != assign[b]:
            div[a][b] += 1
            div[b][a] += 1

all_wits2 = sorted(set(list(coocc.keys()) + list(div.keys()))) or witness_ids

def render_matrix(M, title, max_wits=20):
    wits = all_wits2[:max_wits]
    html = f"<h3>{title}</h3><table style='border-collapse:collapse'>"
    html += "<tr><th style='border:1px solid #ddd;padding:4px'></th>"
    for w in wits:
        html += f"<th style='border:1px solid #ddd;padding:4px'>{w}</th>"
    html += "</tr>"
    for a in wits:
        html += f"<tr><td style='border:1px solid #ddd;padding:4px'><b>{a}</b></td>"
        for b in wits:
            val = 0 if a == b else M[a].get(b, 0)
            html += f"<td style='border:1px solid #ddd;padding:4px;text-align:right'>{val}</td>"
        html += "</tr>"
    html += "</table>"
    display(HTML(html))

render_matrix(coocc, "Co-occurrence (même lecture)")
render_matrix(div, "Divergence (lectures différentes)")

def export_matrix(M, path, wits):
    with path.open("w", encoding="utf-8") as f:
        f.write("wit\t" + "\t".join(wits) + "\n")
        for a in wits:
            row = [str(0 if a == b else M[a].get(b, 0)) for b in wits]
            f.write(a + "\t" + "\t".join(row) + "\n")

co_path = EXPORTS / "cliges_wit_cooccurrence.tsv"
dv_path = EXPORTS / "cliges_wit_divergence.tsv"
export_matrix(coocc, co_path, all_wits2)
export_matrix(div, dv_path, all_wits2)

print("✅ Exports :", co_path, "et", dv_path)

,A,B,C,M,P,R,S,T
A,0,3043,3878,444,3063,3339,2279,1
B,3043,0,3368,413,3394,3145,1909,1
C,3878,3368,0,464,3471,4002,2564,1
M,444,413,464,0,382,438,244,0
P,3063,3394,3471,382,0,3156,1961,0
R,3339,3145,4002,438,3156,0,2422,1
S,2279,1909,2564,244,1961,2422,0,1
T,1,1,1,0,0,1,1,0


,A,B,C,M,P,R,S,T
A,0,2765,2040,212,2652,2445,3085,0
B,2765,0,2527,257,2290,2614,3432,0
C,2040,2527,0,214,2351,1979,2929,0
M,212,257,214,0,261,217,342,0
P,2652,2290,2351,261,0,2565,3343,1
R,2445,2614,1979,217,2565,0,3015,0
S,3085,3432,2929,342,3343,3015,0,0
T,0,0,0,0,1,0,0,0


✅ Exports : exports/cliges_wit_cooccurrence.tsv et exports/cliges_wit_divergence.tsv


## 6) Mini-rapport HTML (carte + tops)

In [31]:
top_lines = density_all.most_common(20)
types_top = type_counts.most_common(20)

mini = "<!doctype html><html><meta charset='utf-8'>"
mini += "<body style='font-family:system-ui;margin:24px'>"
mini += "<h1>Cligès — Rapport rapide (édition TEI unifiée)</h1>"
mini += "<h2>Carte couleur — densité d'apparat par vers</h2>"
mini += "<div style='display:flex;flex-wrap:wrap;max-width:980px'>" + "".join(cells) + "</div>"
mini += "<p><small>Imprimer : Ctrl/Cmd+P → PDF.</small></p>"

mini += "<h2>Top vers (densité)</h2>"
mini += "<table style='border-collapse:collapse'>"
mini += "<tr><th style='border:1px solid #ddd;padding:6px'>vers</th><th style='border:1px solid #ddd;padding:6px'>apparats</th></tr>"
for ln, c in top_lines:
    mini += f"<tr><td style='border:1px solid #ddd;padding:6px'>{ln}</td><td style='border:1px solid #ddd;padding:6px;text-align:right'>{c}</td></tr>"
mini += "</table>"

mini += "<h2>Types (top)</h2>"
mini += "<table style='border-collapse:collapse'>"
mini += "<tr><th style='border:1px solid #ddd;padding:6px'>type</th><th style='border:1px solid #ddd;padding:6px'>count</th></tr>"
for t, c in types_top:
    mini += f"<tr><td style='border:1px solid #ddd;padding:6px'>{t}</td><td style='border:1px solid #ddd;padding:6px;text-align:right'>{c}</td></tr>"
mini += "</table>"

mini += "</body></html>"

rp = EXPORTS / "cliges_rapport_rapide.html"
rp.write_text(mini, encoding="utf-8")
print("✅ Rapport HTML :", rp)


✅ Rapport HTML : exports/cliges_rapport_rapide.html
